In [47]:
import os
import re
import sys
import time
from pathlib import Path
from typing import List, Tuple
import json

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.action_chains import ActionChains
from urllib.parse import urljoin
import random
from bs4 import BeautifulSoup

In [49]:
class kami_crawler:
    def __init__(
            self,
            url: str,
            out_dir: str,
            headless: bool=True,
            wait_s: int=12,
    ) -> None:
        self.url = url
        self.out_dir = out_dir
        self.headless = headless
        self.wait_s = wait_s

        opts = webdriver.ChromeOptions()
        
        chrome_options = os.getenv('CHROME_OPTIONS', '')
        if chrome_options:
            for option in chrome_options.split():
                opts.add_argument(option)
        else:
            # Default options
            if headless:
                opts.add_argument("--headless=new")
            opts.add_argument("--no-sandbox")
            opts.add_argument("--disable-dev-shm-usage")
            opts.add_argument("--disable-gpu")
            opts.add_argument("--window-size=1280,1800")
            opts.add_argument("--disable-extensions")
            opts.add_argument("--disable-plugins")
            opts.add_argument("--disable-images")  # Speed up crawling

        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=opts
        )
        self.wait = WebDriverWait(self.driver, wait_s)

    def _ready(self):
        self.wait.until(lambda d: d.execute_script("return document.readyState") == "complete")
    
    def _visible(self, by, sel):
        return self.wait.until(EC.visibility_of_element_located((by, sel)))

    def extract_kami_list(self):
        self.driver.get(self.url)
        self._ready()

        kami = []

        rows = WebDriverWait(self.driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "tbody[aria-live='polite'] tr"))
        )
        print('Đang lấy danh sách các kami...')

        for row in rows:
            try:
                link_elements = row.find_elements(By.CSS_SELECTOR, "td:nth-of-type(2) a.rel-wiki-page")

                for link_element in link_elements:
                    name = link_element.text.strip()
                    link = link_element.get_attribute("href")

                    #link = urljoin(BASE_URL, link)

                    kami.append({
                        "name": name,
                        "link": link
                    })

            except:
                continue
        print('Đã lấy toàn bộ danh sách của ' + str(len(kami)) + ' kami!')

        return kami

    def parse_kami_data(self, html_content, kami_name):
        soup = BeautifulSoup(html_content, 'html.parser')

        result = {
            'info': {'name': kami_name},
            'skill': [],
            'flavor': ''
        }

        # 1. Duyệt qua tất cả các thẻ <tr> trong bảng
        rows = soup.find_all('tr')
        current_skill_type = None

        for row in rows:
            ths = row.find_all('th', recursive=False)
            tds = row.find_all('td', recursive=False)

            # BẮT TRƯỜNG HỢP FLAVOR TEXT: Dòng chỉ có 1 thẻ <td> và colspan="6"
            if len(tds) == 1 and tds[0].get('colspan') == '6':
                # Dùng separator='\n' để thay thế thẻ <br> thành dấu xuống dòng hợp lệ
                result['flavor'] = tds[0].get_text(separator='\n', strip=True)
                continue

            # XỬ LÝ CÁC DÒNG CÓ TIÊU ĐỀ (TH)
            if ths:
                header_text = ths[0].get_text(strip=True)
                
                # Lấy ảnh nhân vật từ dòng "基本情報"
                if header_text == '基本情報' and len(tds) > 0:
                    img_tag = tds[0].find('img')
                    if img_tag:
                        result['info']['img'] = img_tag['src']
                    continue
                    
                # Xác định các headers của skill để làm mốc
                if 'バースト' in header_text:
                    current_skill_type = 'バースト'
                    continue
                elif 'アビリティ' in header_text:
                    current_skill_type = 'アビリティ'
                    continue
                elif 'アシスト' in header_text:
                    current_skill_type = 'アシスト'
                    continue
                    
                # Cập nhật thông tin cơ bản (Info) - Trừ các dòng header kỹ năng
                if len(ths) == 1 and len(tds) == 1:
                    key = header_text
                    val = tds[0].get_text(strip=True)
                    
                    # Ép kiểu int cho level theo như format mẫu của bạn
                    if key == '最大レベル':
                        val = int(val)
                        
                    result['info'][key] = val
                    continue

            # XỬ LÝ CÁC DÒNG CHỨA DỮ LIỆU SKILL (Chỉ có TD, không có TH)
            if current_skill_type and not ths and len(tds) >= 4:
                skill_dict = {}
                
                # tds[0] là ảnh icon (bỏ qua), tds[1] là tên skill
                skill_dict[current_skill_type] = tds[1].get_text(strip=True)
                
                # tds[2] là điều kiện, nếu trống gán bằng '-'
                condition = tds[2].get_text(strip=True)
                skill_dict['習得条件'] = condition if condition else '-'

                # Phân loại theo skill type vì cột của Ability dài hơn
                if current_skill_type == 'アビリティ' and len(tds) >= 6:
                    skill_dict['使用間隔'] = tds[3].get_text(strip=True)
                    skill_dict['効果時間'] = tds[4].get_text(strip=True)
                    skill_dict['効果'] = tds[5].get_text(strip=True)
                else:
                    # Đối với Burst và Assist
                    skill_dict['効果'] = tds[3].get_text(strip=True)
                    
                result['skill'].append(skill_dict)
            
        return result

    def extract_kami_data(self):
        kami_list = self.extract_kami_list()
        all_data = []
        os.makedirs(self.out_dir, exist_ok=True)
        path = os.path.join(self.out_dir, 'all_kami_data.jsonl')
        
        for i in range(len(kami_list)):
            time.sleep(random.uniform(0.8,1.6))
            self.driver.get(kami_list[i]['link'])
            self._ready()
            kami_name = kami_list[i]['name']
            try:
                print(f'Đang lấy dữ liệu của kami thứ {i + 1}')
                tables = WebDriverWait(self.driver, 10).until(
                    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.h-scrollable")))
                table_html = tables[0].get_attribute("innerHTML")
                kami_data = self.parse_kami_data(table_html, kami_name)
                with open(path, "a", encoding="utf-8") as f:
                    f.write(json.dumps(kami_data, ensure_ascii=False) + "\n")
            except ValueError as e:
                print(f'Gặp lỗi {e} khi lấy data của kami thứ {i}')

In [50]:
if __name__ == "__main__":
    from dotenv import load_dotenv
    load_dotenv()
    test = kami_crawler(r'https://wikiwiki.jp/kamiprodb/%E7%A5%9E%E5%A7%AB/SSR/%E7%81%AB', "kami", False, 10)
    test.extract_kami_data()

Đang lấy danh sách các kami...
Đã lấy toàn bộ danh sách của 106 kami!
Đang lấy dữ liệu của kami thứ 1
Đang lấy dữ liệu của kami thứ 2
Đang lấy dữ liệu của kami thứ 3
Đang lấy dữ liệu của kami thứ 4
Đang lấy dữ liệu của kami thứ 5
Đang lấy dữ liệu của kami thứ 6
Đang lấy dữ liệu của kami thứ 7
Đang lấy dữ liệu của kami thứ 8
Đang lấy dữ liệu của kami thứ 9
Đang lấy dữ liệu của kami thứ 10
Đang lấy dữ liệu của kami thứ 11
Đang lấy dữ liệu của kami thứ 12
Đang lấy dữ liệu của kami thứ 13
Đang lấy dữ liệu của kami thứ 14
Đang lấy dữ liệu của kami thứ 15
Đang lấy dữ liệu của kami thứ 16
Đang lấy dữ liệu của kami thứ 17
Đang lấy dữ liệu của kami thứ 18
Đang lấy dữ liệu của kami thứ 19
Đang lấy dữ liệu của kami thứ 20
Đang lấy dữ liệu của kami thứ 21
Đang lấy dữ liệu của kami thứ 22
Đang lấy dữ liệu của kami thứ 23
Đang lấy dữ liệu của kami thứ 24
Đang lấy dữ liệu của kami thứ 25
Đang lấy dữ liệu của kami thứ 26
Đang lấy dữ liệu của kami thứ 27
Đang lấy dữ liệu của kami thứ 28
Đang lấy dữ liệ

In [40]:
print(kami_list)

[{'name': '［競走路の女王］オシリス', 'link': 'https://wikiwiki.jp/kamiprodb/%EF%BC%BB%E7%AB%B6%E8%B5%B0%E8%B7%AF%E3%81%AE%E5%A5%B3%E7%8E%8B%EF%BC%BD%E3%82%AA%E3%82%B7%E3%83%AA%E3%82%B9'}, {'name': '［HELIX］ニケ', 'link': 'https://wikiwiki.jp/kamiprodb/%EF%BC%BBHELIX%EF%BC%BD%E3%83%8B%E3%82%B1'}, {'name': '［春待ちの温もり］サマエル', 'link': 'https://wikiwiki.jp/kamiprodb/%EF%BC%BB%E6%98%A5%E5%BE%85%E3%81%A1%E3%81%AE%E6%B8%A9%E3%82%82%E3%82%8A%EF%BC%BD%E3%82%B5%E3%83%9E%E3%82%A8%E3%83%AB'}, {'name': '［正月福娘］フィア', 'link': 'https://wikiwiki.jp/kamiprodb/%EF%BC%BB%E6%AD%A3%E6%9C%88%E7%A6%8F%E5%A8%98%EF%BC%BD%E3%83%95%E3%82%A3%E3%82%A2'}, {'name': 'ヒュプノス［心想昇華］', 'link': 'https://wikiwiki.jp/kamiprodb/%E3%83%92%E3%83%A5%E3%83%97%E3%83%8E%E3%82%B9%EF%BC%BB%E5%BF%83%E6%83%B3%E6%98%87%E8%8F%AF%EF%BC%BD'}, {'name': '［省察の先に］茨木童子', 'link': 'https://wikiwiki.jp/kamiprodb/%EF%BC%BB%E7%9C%81%E5%AF%9F%E3%81%AE%E5%85%88%E3%81%AB%EF%BC%BD%E8%8C%A8%E6%9C%A8%E7%AB%A5%E5%AD%90'}, {'name': '［朧月の玉兎］イリス', 'link': 'https://wikiwiki.jp/k

In [41]:
import pandas as pd
df = pd.DataFrame(kami_list)
print(df)

              name                                               link
0     ［競走路の女王］オシリス  https://wikiwiki.jp/kamiprodb/%EF%BC%BB%E7%AB%...
1        ［HELIX］ニケ  https://wikiwiki.jp/kamiprodb/%EF%BC%BBHELIX%E...
2    ［春待ちの温もり］サマエル  https://wikiwiki.jp/kamiprodb/%EF%BC%BB%E6%98%...
3        ［正月福娘］フィア  https://wikiwiki.jp/kamiprodb/%EF%BC%BB%E6%AD%...
4      ヒュプノス［心想昇華］  https://wikiwiki.jp/kamiprodb/%E3%83%92%E3%83%...
..             ...                                                ...
101     不動明王［神化覚醒］  https://wikiwiki.jp/kamiprodb/%E4%B8%8D%E5%8B%...
102          アマテラス  https://wikiwiki.jp/kamiprodb/%E3%82%A2%E3%83%...
103    アマテラス［神化覚醒］  https://wikiwiki.jp/kamiprodb/%E3%82%A2%E3%83%...
104            アレス  https://wikiwiki.jp/kamiprodb/%E3%82%A2%E3%83%...
105      アレス［神化覚醒］  https://wikiwiki.jp/kamiprodb/%E3%82%A2%E3%83%...

[106 rows x 2 columns]


In [5]:
print(test_detail)

{'base': {'name': 'ソル', 'image': 'https://xn--hckqz0e9cygq471ahu9b.xn--wiki-4i9hs14f.com/index.php?plugin=ref&page=%E3%82%BD%E3%83%AB&src=%E3%82%BD%E3%83%AB2022.png', 'レアリティ': 'SSR', 'MaxHP/覚醒後': '1690', 'Max攻撃力': '6500', 'skills': [['ホワイト・プロミネンス+', '第3段階限界突破により解放される', '光属性ダメージ(特大)★性能UP'], ['太陽光炉+', '55', '6T', '-', '味方全体のHP回復(上限1500)/状態異常を1つ消去'], ['アールヴレズル', '-', '9T', '-', '敵単体に光属性ダメージ/敵の強化効果を1つ解除'], ['アールヴレズル+', '75', '8T', '-', '敵単体に光属性ダメージ/敵の強化効果を1つ解除★ターン短縮'], ['カルドルーチェ', '45', '7T', '180秒', '敵全体の攻撃DOWN(C枠/-20%)']], 'flavor': '恒星に匹敵するパワーを内包した女神。\nその明るい様子とは裏腹に苛烈な運命を背負っている。'}, 'awaken': {'name': 'ソル[神化覚醒]', 'image': 'https://xn--hckqz0e9cygq471ahu9b.xn--wiki-4i9hs14f.com/index.php?plugin=ref&page=%E3%82%BD%E3%83%AB&src=sol.png', 'MaxHP/覚醒後': '2500', 'Max攻撃力': '7500', 'skills': [['太陽光炉++', '65', '6T', '3T', '味方全体のHP回復(上限1800)/状態異常を2つ消去/リジェネ付与(250)'], ['アールヴレズル+', '-', '8T', '-', '敵単体に光属性ダメージ/敵の強化効果を1つ解除★ターン短縮'], ['アールヴレズル++', '75', '7T', '-', '敵全体に光属性ダメージ/敵の強化効果を1つ解除★ターン短縮'], ['カルドルー